# Part A: Create a notebook called classification

In [3]:
#initial data prep section. Read, clean, and create sets
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

airbnb = pd.read_csv("listings.csv")
crime = pd.read_csv("Crime_Data_from_2020_to_Present.csv")

print(airbnb.head())
print(crime.head())

airbnb['price'] = airbnb['price'].replace(r'[\$,]', '', regex=True).astype(float)
airbnb = airbnb.drop_duplicates()
airbnb = airbnb.dropna(subset=['price', 'bedrooms', 'bathrooms'])
airbnb['review_scores_rating'] = airbnb['review_scores_rating'].fillna(airbnb['review_scores_rating'].mean())

crime_count = len(crime)
airbnb['crime_count'] = crime_count

airbnb['price_category'] = pd.qcut(airbnb['price'], 3, labels=['Low', 'Medium', 'High'])

X = airbnb[['bedrooms', 'bathrooms', 'number_of_reviews', 'availability_365', 'review_scores_rating', 'crime_count']]
y = airbnb['price_category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

C:\Users\S553902\AppData\Local\Temp\ipykernel_26584\157028882.py:6: DtypeWarning: Columns (0: host_since) have mixed types. Specify dtype option on import or set low_memory=False.
  airbnb = pd.read_csv("listings.csv")


                    id                                               name  \
0   670339032744709144     Westwood lovely three bedrooms three bathrooms   
1             37014494      Spanish style lower duplex near Beverly Hills   
2  1024835174766068422                        Charming Beverly Hills Home   
3   850744632375448560                   Tianpu's warm room with bathroom   
4   953950676345326970  Santa Monica apt, free parking, steps to the b...   

     host_id host_name host_since  host_response_time  host_response_rate  \
0    4780152      Moon   20/01/13  within a few hours                0.96   
1  278288178       Ida   22/07/19                 NaN                 NaN   
2  513813179     Tiana   08/05/23        within a day                0.60   
3  432956623       Dan   22/11/21  a few days or more                0.20   
4  528669205   Farkhat   29/07/23      within an hour                1.00   

  host_is_superhost neighbourhood_cleansed neighbourhood_group_cleansed  .

#### do a decision tree on X and y. Compute metrics.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

**Comment on the results.**

The decision tree model achieved an accuracy of approximately 56%, indicating that it is able to correctly classify Airbnb listings into price categories slightly better than random guessing which was about 33%. 

Looking at the classification report, the model performs the best when predicting high priced listings with an f1 score of 0.67, suggesting that these listings are easier to identify based on the selected features. The model also performs reasonably well for low priced listings with an f1 score of 0.58.

However, the model struggles more with medium priced listings which have a lower f1 score of 0.43. This suggests that medium priced listings may share characteristics with both low and high priced listings, making them harder to classify accurately. 

Overall, the results show that the model is capturing some meaningful patterns in the data, especially for extreme price categories, but performance is still moderate and could be improved with additional features.

In [9]:
#see if you can do better using SVM or some other multi-classifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

svm_model = SVC()
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm, zero_division=0))

SVM Accuracy: 0.3301023155627356
              precision    recall  f1-score   support

        High       0.00      0.00      0.00      2522
         Low       0.33      1.00      0.50      2452
      Medium       0.00      0.00      0.00      2454

    accuracy                           0.33      7428
   macro avg       0.11      0.33      0.17      7428
weighted avg       0.11      0.33      0.16      7428



In [10]:
#do a final evaluation with the test set
print("Final Model Comparison:\n")

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred))
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

Final Model Comparison:

Decision Tree Accuracy: 0.560446957458266
SVM Accuracy: 0.3301023155627356


**Look at the parameters you found and discuss what you have learned.**

The classficiation models were used to predict whether an Airbnb listing falls into low, medium, or high price categories. The decision tree model achieved an accuracy of 56% indicating that it was able to capture meaningful patterns in the data. It performed best on high priced listings and reasonably well on low priced listings but struggled with medium priced listings.

In contrast, the SVM model perfomed significantly worse, with an accuracy of 33%, which is roughly equivalent to random guessing for three classes. The classification report shows that the SVM model predicted only the low category, with a recall of 1.00 for low-priced listings and 0.00 for both medium and high categories. This indicates that the model failed to distinguish between the different price groups and instead defaulted to predicting a single class. 

Comparing the two models, the decision tree clearly outperformed the SVM model. The poor performance of the SVM suggests that it was not well-suited for this dataset in its current form, likely due to the feature scaling or the complexity of the data.

Overall, the results show that while property characteristics like bedrooms and bathrooms help classify price categories, they are not sufficient for strong predictive performance. The difficulty in distinguishing between medium and high-priced listings suggests that additional features like location, amenities, or neighborhood factors like crime rates are important for improving classification accuracy.

# Part B

**Project overview, key insights, and conclusions**
This project explores predicting Airbnb listing prices in Los Angeles using both regression and classification approaches, as well as incorporating external data such as crime statistics.

In the regression notebook, a linear regression model was used to predict the exact nightly price of Airbnb listings. The results showed that property characteristics such as bedrooms and bathrooms have a noticeable impact on price. The initial model achieved an R² of approximately 0.20, which improved to about 0.29 after adding additional features. While this indicates some predictive ability, it also shows that a large portion of price variation is influenced by factors not included in the model.

In the classification notebook, the problem was reframed by grouping prices into categories (Low, Medium, High). A decision tree model achieved an accuracy of about 56%, showing moderate success in distinguishing between price categories, especially for low and high-priced listings. However, the model struggled with medium-priced listings, suggesting overlap between categories.

An SVM model was also tested but performed poorly, with an accuracy of about 33%, predicting only one class. This indicates that the model was not well-suited for the dataset without further preprocessing such as feature scaling.

Overall, the results from both notebooks suggest that while property characteristics provide some predictive power, they are not sufficient for highly accurate predictions. Additional features such as location, amenities, and neighborhood-level factors like crime rates are likely important for improving model performance.

This project highlights the importance of feature selection, data preparation, and model choice in machine learning tasks, and provides a foundation for further improvements in future work.